# The Binomial Option Pricing Model

A from-scratch implementation of the binomial tree model for pricing European and American options,
including Greeks computation, convergence analysis, and comparison with the Black-Scholes-Merton formula.

## 1. What Are Options, and Why Do We Need a Model?

Before diving into math, let us build intuition.

An **option** is a contract that gives you the *right* (but not the obligation) to buy or sell a stock at a predetermined price. Think of it like **insurance**:

- A **call option** is like buying the right to purchase a house at today's price, even if the market goes up. You would gladly pay a small premium for that right if you think prices might rise.
- A **put option** is like buying insurance on a stock you own. If the stock crashes, the put option pays you the difference.

The fundamental question is: **how much should this insurance cost?**

### Key Terminology

| Term | Meaning | Real-World Analogy |
|------|---------|-------------------|
| **Strike price** ($K$) | The pre-agreed price to buy/sell | The deductible on your insurance |
| **Premium** | What you pay for the option | The insurance premium |
| **Expiry** ($T$) | When the option expires | When your insurance policy ends |
| **Underlying** ($S$) | The stock the option is based on | The house you are insuring |
| **In the money (ITM)** | Option has positive intrinsic value | Your insurance is "activated" |
| **Out of the money (OTM)** | Option has no intrinsic value | Your insurance has not kicked in |
| **At the money (ATM)** | Stock price equals strike price | Right at the threshold |

The binomial model is the simplest framework that answers the pricing question correctly. It was introduced by Cox, Ross, and Rubinstein (1979) and remains one of the most intuitive and powerful tools in quantitative finance.

> **Key Concept:** You do not need to know the probability that a stock goes up or down to price an option. All you need is the **no-arbitrage** principle: there should be no way to make free money. This single idea determines the option price uniquely.

### Why Not Just Use Expected Values?

Your first instinct might be: "Estimate the probability the stock goes up, compute the expected payoff, and that is the option price." This approach fails because:

1. **Different investors disagree** about the probability of the stock going up. A bull might say 70%, a bear might say 30%. They cannot both be right about the price.
2. **Risk preferences matter.** Even if investors agree on probabilities, they would demand different compensation for bearing risk.
3. **The market already embeds probabilities and risk preferences** into the stock price. We can extract what we need from market prices alone.

The binomial model sidesteps all of this by using a far more powerful idea: **replication**.

## 2. The Simplest Possible World: One Period, Two Outcomes

Imagine the simplest possible stock market:

- Today, the stock price is $S_0 = \$100$.
- Tomorrow, the stock either goes **up** to $\$120$ or **down** to $\$80$.
- There is a risk-free bond that earns 5% interest.

This is like a **coin flip** for stock prices --- heads it goes up, tails it goes down. We do not even need to know if the coin is fair!

### Setting Up the Problem

We describe the stock's movement with two numbers:

- **Up factor** $u = 1.2$: if the stock goes up, it is multiplied by $u$, giving $S_u = 100 \times 1.2 = 120$.
- **Down factor** $d = 0.8$: if it goes down, $S_d = 100 \times 0.8 = 80$.

Now consider a **European call option** with strike $K = \$100$:

```
                    S_u = 120  -->  Payoff = max(120 - 100, 0) = $20
                   /
    S_0 = 100 ----
                   \
                    S_d = 80   -->  Payoff = max(80 - 100, 0) = $0
```

The payoff is simple: if the stock ends above the strike, you profit by the difference. If not, the option is worthless.

> **Key Concept:** The payoff of a call is $\max(S_T - K, 0)$. You never lose more than the premium you paid. That is what makes it a "right" rather than an "obligation." Similarly, a put pays $\max(K - S_T, 0)$.

### What Should This Option Cost?

Your first instinct might be: "Figure out the probability that the stock goes up, compute the expected payoff, and discount it." This is tempting but **wrong** --- different investors have different beliefs about the probability, so they would disagree on the price.

The binomial model uses a much cleverer approach: **replication**.

## 3. The No-Arbitrage Argument

### The Key Insight: Replication

Here is the brilliant idea behind option pricing: we can **replicate** the option's payoff exactly by holding a carefully chosen portfolio of stock and bonds.

Suppose we hold $\Delta$ shares of stock and $B$ dollars in bonds. At expiry:

- **Up state**: our portfolio is worth $\Delta \times 120 + B \times 1.05$
- **Down state**: our portfolio is worth $\Delta \times 80 + B \times 1.05$

We want this portfolio to match the call's payoff in both states:

$$120\Delta + 1.05B = 20 \quad \text{(up state)}$$
$$80\Delta + 1.05B = 0 \quad \text{(down state)}$$

### Solving the System (Worked Example)

**Step 1**: Subtract the second equation from the first:

$$120\Delta - 80\Delta = 20 - 0 \implies 40\Delta = 20 \implies \Delta = 0.5$$

So we need to hold **half a share** of stock. This $\Delta$ is called the **hedge ratio** or **delta** of the option.

**Step 2**: Plug $\Delta = 0.5$ back into the second equation:

$$80 \times 0.5 + 1.05B = 0 \implies 40 + 1.05B = 0 \implies B = -38.10$$

The negative $B$ means we **borrow** \$38.10 at the risk-free rate.

**Step 3**: The option price must equal the cost of this replicating portfolio:

$$C = \Delta \times S_0 + B = 0.5 \times 100 + (-38.10) = \$11.90$$

> **Key Concept:** If the option were priced differently from \$11.90, you could make free money (arbitrage) by either buying the cheap side and selling the expensive side. The market will not allow this, so the price must be \$11.90.

### Why This Works: The Arbitrage Argument in Detail

Suppose someone is selling the call for \$10 (too cheap). You could:
1. Buy the call for \$10.
2. Sell the replicating portfolio (short 0.5 shares, lend \$38.10), collecting \$11.90.
3. Pocket the \$1.90 difference **risk-free**.

At expiry, the call and the replicating portfolio have identical payoffs, so they cancel out. You keep \$1.90 no matter what the stock does. This is arbitrage --- free money --- and in efficient markets, it gets exploited instantly until the prices align.

> **Important:** The replication argument does not use the real-world probability of the stock going up or down. Two investors who completely disagree about where the stock is headed will still agree on the option price. This is the profound insight of the binomial model.

### What About the Put?

Using the same approach for a put with $K = 100$:
- Up state payoff: $\max(100 - 120, 0) = 0$
- Down state payoff: $\max(100 - 80, 0) = 20$

Solving the replication equations gives a different $\Delta$ (negative, since puts move opposite to the stock) and a different portfolio cost.

## 4. Risk-Neutral Probability: A Shortcut That Reveals Deep Truth

The replication argument above works perfectly, but there is an elegant shortcut.

Instead of solving two equations, we can define a special probability $q$ --- called the **risk-neutral probability** --- and compute the option price as a simple expected value.

### What is $q$?

The risk-neutral probability $q$ is the probability of an up-move that would make an investor **indifferent between the stock and the bond**. In other words, under $q$, the expected return on the stock equals the risk-free rate:

$$q \cdot S_u + (1-q) \cdot S_d = S_0 \cdot e^{rT}$$

Solving for $q$:

$$q = \frac{e^{rT} - d}{u - d}$$

where:
- $e^{rT}$ is the growth factor of a risk-free bond over one period
- $u$ is the up factor (how much the stock multiplies if it goes up)
- $d$ is the down factor (how much it multiplies if it goes down)

### Worked Example

With our numbers: $u = 1.2$, $d = 0.8$, $r = 0.05$, $T = 1$:

$$q = \frac{e^{0.05} - 0.8}{1.2 - 0.8} = \frac{1.0513 - 0.8}{0.4} = \frac{0.2513}{0.4} \approx 0.6282$$

### The Pricing Formula

With $q$ in hand, the option price is simply the **discounted expected payoff under the risk-neutral probability**:

$$C = e^{-rT}\bigl[q \cdot f_u + (1-q) \cdot f_d\bigr]$$

where $f_u$ and $f_d$ are the option payoffs in the up and down states.

$$C = e^{-0.05}\bigl[0.6282 \times 20 + 0.3718 \times 0\bigr] = 0.9512 \times 12.564 = \$11.95$$

(The small difference from our earlier \$11.90 is due to rounding --- with exact arithmetic, both methods give identical results.)

### The Deep Meaning

The risk-neutral probability $q$ is **not** the real-world probability of the stock going up. It is a mathematical construction that allows us to price options as if investors did not care about risk. In this fictional risk-neutral world:

- Every asset earns the risk-free rate on average.
- We can price by taking expectations and discounting at the risk-free rate.

This works because the option price is determined by replication, not by beliefs about the future.

> **Common Mistake:** Students often think $q$ is the probability the stock goes up. It is not. The real probability might be 0.7 or 0.3 or anything --- it does not matter for pricing. What matters is the no-arbitrage condition.

| Concept | Real World | Risk-Neutral World |
|---------|-----------|-------------------|
| Stock expected return | $\mu$ (e.g., 8%) | $r$ (e.g., 5%) |
| Probability of up-move | $p$ (unknown/subjective) | $q$ (derived from no-arbitrage) |
| How to price | Requires risk adjustment | Simple: $e^{-rT} \mathbb{E}^q[\text{payoff}]$ |
| Who uses it | Economists studying returns | Option traders pricing contracts |

> **Key Concept:** Risk-neutral pricing is not an assumption about the world --- it is a mathematical technique. The real world is full of risk-averse investors. But because options can be replicated, their prices are the same as they would be in a risk-neutral world. This is one of the most powerful results in financial economics.

Now let us implement the one-period model and verify our hand calculations.

In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats, optimize
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
rng = np.random.default_rng(SEED)

ATOL = 1e-10
RTOL = 1e-6

PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

In [ ]:
def one_period_binomial(S0, K, r, T, u, d, option_type='call'):
    """Price a European option in a one-period binomial model.
    
    Parameters
    ----------
    S0 : float -- current stock price
    K  : float -- strike price
    r  : float -- risk-free rate (continuous compounding)
    T  : float -- time to maturity
    u  : float -- up factor
    d  : float -- down factor
    option_type : str -- 'call' or 'put'
    
    Returns
    -------
    price : float
    q     : float -- risk-neutral probability
    """
    dt = T
    disc = np.exp(-r * dt)           # discount factor: converts future $ to today's $
    q = (np.exp(r * dt) - d) / (u - d)  # risk-neutral probability of up-move
    
    # Stock prices in each state
    Su = u * S0
    Sd = d * S0
    
    # Option payoffs in each state
    if option_type == 'call':
        fu = max(Su - K, 0)  # call pays max(S - K, 0)
        fd = max(Sd - K, 0)
    else:
        fu = max(K - Su, 0)  # put pays max(K - S, 0)
        fd = max(K - Sd, 0)
    
    # Risk-neutral pricing: discounted expected payoff under q
    price = disc * (q * fu + (1 - q) * fd)
    return price, q


# --- Verify our hand-worked example ---
S0, K, r, T = 100, 100, 0.05, 1.0
u, d = 1.2, 0.8

call_price, q = one_period_binomial(S0, K, r, T, u, d, 'call')
put_price, _  = one_period_binomial(S0, K, r, T, u, d, 'put')

print(f"One-period binomial model")
print(f"  u = {u}, d = {d}, q = {q:.6f}")
print(f"  Call price: {call_price:.4f}")
print(f"  Put price:  {put_price:.4f}")
print(f"  Put-call parity check: C - P = {call_price - put_price:.4f}, "
      f"S - K*exp(-rT) = {S0 - K * np.exp(-r * T):.4f}")

The code confirms our hand calculation: $q \approx 0.6282$ and the call is priced at about $\$11.95$.

Notice also that **put-call parity** holds: $C - P = S_0 - K e^{-rT}$. This is a model-independent relationship that we explore in a separate notebook.

> **Key Concept:** Put-call parity is a consistency check. If your call and put prices do not satisfy $C - P = S_0 - Ke^{-rT}$, something is wrong with your model or your implementation.

## 5. Extending to Multiple Periods: The Binomial Tree

One period is too crude. Real option pricing needs many periods to capture the range of possible stock prices.

### The Idea: Stack Coin Flips

Imagine flipping our up/down coin many times. After each flip, the stock either goes up by factor $u$ or down by factor $d$. After $N$ flips, the stock has taken one of $2^N$ possible paths.

The beautiful thing is that the tree **recombines**: an up-then-down move gives the same price as a down-then-up move ($S_0 \cdot u \cdot d = S_0 \cdot d \cdot u$). This reduces the number of distinct terminal prices from $2^N$ to just $N + 1$.

### A Two-Period Example (Worked by Hand)

Let us work through a 2-period tree with $S_0 = 100$, $u = 1.2$, $d = 0.8$, $r = 0.05$, $\Delta t = 0.5$ (six months per period).

**Step 1: Build the stock price tree forward.**

```
                              S_uu = 100 * 1.2 * 1.2 = $144
                             /
            S_u = 100 * 1.2 = $120
           /                 \
  S_0 = $100                  S_ud = 100 * 1.2 * 0.8 = $96
           \                 /
            S_d = 100 * 0.8 = $80
                             \
                              S_dd = 100 * 0.8 * 0.8 = $64
```

**Step 2: Compute terminal payoffs for a call with $K = 100$.**

- $f_{uu} = \max(144 - 100, 0) = \$44$
- $f_{ud} = \max(96 - 100, 0) = \$0$
- $f_{dd} = \max(64 - 100, 0) = \$0$

**Step 3: Work backwards one step.** The risk-neutral probability for each period is:

$$q = \frac{e^{0.05 \times 0.5} - 0.8}{1.2 - 0.8} = \frac{1.0253 - 0.8}{0.4} = 0.5633$$

At the up node ($S_u = 120$):

$$f_u = e^{-0.025}[0.5633 \times 44 + 0.4367 \times 0] = 0.9753 \times 24.79 = \$24.18$$

At the down node ($S_d = 80$):

$$f_d = e^{-0.025}[0.5633 \times 0 + 0.4367 \times 0] = \$0$$

**Step 4: Work backwards to today.**

$$C = e^{-0.025}[0.5633 \times 24.18 + 0.4367 \times 0] = 0.9753 \times 13.62 = \$13.28$$

> **Key Concept:** The backward induction process is the heart of the binomial model. We start with known payoffs at expiry and work backwards, computing the discounted risk-neutral expected value at each node. This "rolling back" through the tree gives us the price today.

### Scaling Up

With 2 periods, we have 3 terminal prices. With 100 periods, we have 101 terminal prices. With 1000, we have 1001. The computation scales linearly --- each step only requires combining adjacent values from the previous step.

| Periods | Terminal Nodes | Paths (non-recombining) |
|---------|---------------|------------------------|
| 2 | 3 | 4 |
| 10 | 11 | 1,024 |
| 100 | 101 | ~$10^{30}$ |
| 1000 | 1001 | ~$10^{301}$ |

Without recombination, even 100 periods would be intractable. With it, we can easily handle thousands.

## 6. The CRR Parameterization: Connecting to Reality

So far, we chose $u$ and $d$ arbitrarily. But for the binomial model to approximate real stock behavior, we need to choose them carefully.

### Why CRR?

Cox, Ross, and Rubinstein (1979) showed that if we set:

$$u = e^{\sigma\sqrt{\Delta t}}, \quad d = e^{-\sigma\sqrt{\Delta t}} = \frac{1}{u}$$

where:
- $\sigma$ is the **annualized volatility** of the stock (e.g., $\sigma = 0.2$ means 20% annual volatility)
- $\Delta t = T/N$ is the length of each time step

then as $N \to \infty$, the binomial model converges to the Black-Scholes-Merton model. In other words, the discrete coin-flip model becomes the continuous random-walk model.

### Intuition for CRR Parameters

Think of it this way:

- $\sigma$ measures how much the stock **wiggles** per year.
- Over a short period $\Delta t$, the wiggle is proportional to $\sigma\sqrt{\Delta t}$.
- The up factor $u = e^{\sigma\sqrt{\Delta t}}$ captures one wiggle up.
- The down factor $d = 1/u$ ensures the tree recombines (up-then-down = down-then-up).

> **Key Concept:** The $\sqrt{\Delta t}$ scaling is crucial. It comes from the mathematical property that a random walk's standard deviation grows as the square root of time. If you double the time period, the range of outcomes grows by $\sqrt{2}$, not by 2. This is the same reason why annual volatility is roughly $\sqrt{252}$ times daily volatility.

### Worked Example

For $\sigma = 0.20$, $T = 1$ year, $N = 4$ steps ($\Delta t = 0.25$):

$$u = e^{0.20 \sqrt{0.25}} = e^{0.10} = 1.1052$$
$$d = 1/u = 0.9048$$
$$q = \frac{e^{0.05 \times 0.25} - 0.9048}{1.1052 - 0.9048} = \frac{1.0126 - 0.9048}{0.2004} = 0.5374$$

Let us build and visualize a small CRR tree to see this structure.

In [ ]:
def build_crr_tree(S0, sigma, r, T, N):
    """Build a CRR binomial tree and return stock price lattice.
    
    Returns
    -------
    S : list of arrays -- S[n][j] is the stock price at step n, node j
    u, d, q : float -- CRR parameters
    """
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))   # up factor from CRR
    d = 1.0 / u                       # down factor (ensures recombination)
    q = (np.exp(r * dt) - d) / (u - d)  # risk-neutral probability
    
    # Build stock price tree: at step n, there are n+1 nodes
    S = []
    for n in range(N + 1):
        j = np.arange(n + 1)
        S_n = S0 * u**j * d**(n - j)  # node j has j up-moves and (n-j) down-moves
        S.append(S_n)
    
    return S, u, d, q


# Build and display a small tree
S0, sigma, r, T = 100, 0.2, 0.05, 1.0
N_demo = 4
S_tree, u, d, q = build_crr_tree(S0, sigma, r, T, N_demo)

print(f"CRR parameters: u = {u:.6f}, d = {d:.6f}, q = {q:.6f}")
print(f"\nStock price tree (N={N_demo}):")
for n in range(N_demo + 1):
    prices = ', '.join(f'{s:.2f}' for s in S_tree[n])
    print(f"  Step {n}: [{prices}]")

Notice how the tree fans out: step 0 has 1 node, step 1 has 2, and so on up to step 4 with 5 nodes. The highest node at step 4 represents 4 consecutive up-moves; the lowest represents 4 consecutive down-moves.

> **Key Concept:** The CRR tree recombines, meaning up-then-down gives the same price as down-then-up. This makes the tree computationally efficient: $N+1$ terminal nodes instead of $2^N$ paths. Without recombination, even 50 steps would require over $10^{15}$ path evaluations.

## 7. Pricing European Options on the Tree

Now we have all the ingredients for pricing. The algorithm is:

1. **Build the tree forward** to get stock prices at every node.
2. **Compute terminal payoffs** at step $N$: $\max(S_T - K, 0)$ for calls, $\max(K - S_T, 0)$ for puts.
3. **Work backwards** using the risk-neutral pricing formula at each node:

$$f_{n,j} = e^{-r\Delta t}\bigl[q\, f_{n+1,j+1} + (1-q)\, f_{n+1,j}\bigr]$$

In words: the price at any node equals "the discounted weighted average of the two possible future values, where the weights are the risk-neutral probabilities."

For European options, that is all there is. We never exercise early, so the value at each node is purely the discounted expected continuation value.

> **Key Concept:** Backward induction is essentially dynamic programming. We solve a big problem (pricing today) by breaking it into small problems (pricing at each node) and working backwards from the known answer (terminal payoffs).

### Implementation Trick: Work with a 1-D Array

Instead of storing the entire tree (which would require $O(N^2)$ memory), we can use a single array of length $N+1$ and overwrite it at each step. At step $n$, positions 0 through $n$ contain the option values. This saves memory and is much faster.

The following code implements both the BSM closed-form solution (for reference) and the binomial European pricer.

In [ ]:
def bsm_price(S0, K, r, T, sigma, option_type='call'):
    """Analytical Black-Scholes-Merton price for reference."""
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == 'call':
        return S0 * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * stats.norm.cdf(-d2) - S0 * stats.norm.cdf(-d1)


def binomial_european(S0, K, r, T, sigma, N, option_type='call'):
    """Price a European option using the CRR binomial model.
    
    Uses efficient backward induction with a 1-D array.
    """
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1.0 / u
    q = (np.exp(r * dt) - d) / (u - d)
    disc = np.exp(-r * dt)  # one-period discount factor
    
    # Terminal stock prices at step N
    j = np.arange(N + 1)
    ST = S0 * u**j * d**(N - j)
    
    # Terminal payoffs (the known values we start from)
    if option_type == 'call':
        V = np.maximum(ST - K, 0.0)
    else:
        V = np.maximum(K - ST, 0.0)
    
    # Backward induction: work from step N-1 back to step 0
    for n in range(N - 1, -1, -1):
        # At step n, node j gets the discounted weighted average of nodes j and j+1 at step n+1
        V = disc * (q * V[1:n+2] + (1 - q) * V[0:n+1])
    
    return V[0]  # the option price today (step 0, node 0)

Let us compare the binomial price against the exact Black-Scholes-Merton formula for increasing numbers of steps. As $N$ grows, the binomial price should converge to BSM.

In [ ]:
# Compare binomial with BSM across different N
S0, K, r, T, sigma = 100, 100, 0.05, 1.0, 0.2

bsm_call_ref = bsm_price(S0, K, r, T, sigma, 'call')
bsm_put_ref  = bsm_price(S0, K, r, T, sigma, 'put')

for N in [10, 50, 100, 500, 1000]:
    bin_call = binomial_european(S0, K, r, T, sigma, N, 'call')
    bin_put  = binomial_european(S0, K, r, T, sigma, N, 'put')
    print(f"N = {N:5d}  |  Call: {bin_call:.6f} (err {abs(bin_call - bsm_call_ref):.2e})  |  "
          f"Put: {bin_put:.6f} (err {abs(bin_put - bsm_put_ref):.2e})")

print(f"\nBSM ref  |  Call: {bsm_call_ref:.6f}                     |  Put: {bsm_put_ref:.6f}")

The error shrinks as $N$ increases. With just 100 steps, we are already within a few cents of the BSM value. With 1000 steps, the error is negligible.

| N | Approximate Error |
|---|---|
| 10 | ~$0.10 |
| 50 | ~$0.02 |
| 100 | ~$0.01 |
| 1000 | ~$0.001 |

> **Key Concept:** The binomial model is a **discrete approximation** to the continuous BSM model. As the number of steps increases, each step becomes shorter, the coin flips become more frequent, and the model approaches continuous-time reality. This is analogous to how a polygon with more sides approaches a circle.

Let us visualize this convergence.

In [ ]:
# Convergence plot: binomial vs BSM as N -> infinity
N_values = np.arange(5, 301)
bin_prices = np.array([binomial_european(S0, K, r, T, sigma, int(N), 'call') for N in N_values])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(N_values, bin_prices, color=PRIMARY, alpha=0.7, linewidth=0.8)
axes[0].axhline(bsm_call_ref, color=SECONDARY, linestyle='--', linewidth=2, label=f'BSM = {bsm_call_ref:.4f}')
axes[0].set_xlabel('Number of Steps (N)')
axes[0].set_ylabel('Call Price')
axes[0].set_title('Binomial Convergence to BSM')
axes[0].legend()

errors = np.abs(bin_prices - bsm_call_ref)
axes[1].semilogy(N_values, errors, color=PRIMARY, alpha=0.7, linewidth=0.8)
axes[1].set_xlabel('Number of Steps (N)')
axes[1].set_ylabel('Absolute Error')
axes[1].set_title('Convergence Error (log scale)')

plt.tight_layout()
plt.show()

The left plot shows the binomial price oscillating around and converging to the BSM price. The right plot (log scale) confirms that the error decreases roughly as $O(1/N)$.

Notice the **zigzag pattern** --- even and odd $N$ approach the BSM value from opposite sides. This is the "even-odd oscillation" we will explore in Section 11.

## 8. American Options: The Right to Exercise Early

### European vs. American: What is the Difference?

| Feature | European | American |
|---------|----------|----------|
| Exercise | Only at expiry $T$ | Any time up to $T$ |
| Pricing | Backward induction only | Backward induction + early exercise check |
| Call on non-dividend stock | Same as American | Same as European |
| Put | Cheaper than American put | More expensive (early exercise has value) |

### Why Would You Exercise Early?

**For a put:** Suppose you hold a put with strike \$100 and the stock has crashed to \$1. Your put is worth at least \$99 if you exercise now. But if you wait, the stock might recover (reducing your payoff) or stay low (giving you roughly the same payoff, but later and thus less valuable due to time value of money). You would rather have \$99 today to invest at the risk-free rate.

Think of it like holding a winning lottery ticket that slowly decays in value. At some point, it is better to cash it in than to keep waiting.

**For a call on a non-dividend stock:** Early exercise is **never** optimal. Why? Because:
1. You pay $K$ now instead of $K$ later, losing the interest you could earn on $K$ in the meantime.
2. The "insurance" value of the call (protecting you if the stock drops) is always worth something.
3. You can always sell the call in the market for at least its intrinsic value.

> **Important:** Early exercise of calls *can* be optimal if the stock pays dividends. Just before a large dividend payment, the stock price drops by the dividend amount. Exercising the call to capture the dividend can be worthwhile.

### How the Algorithm Changes

For American options, at each node during backward induction, we add one check:

$$V_{n,j} = \max\!\left(\underbrace{e^{-r\Delta t}[q\, V_{n+1,j+1} + (1-q)\, V_{n+1,j}]}_{\text{continue holding}},\; \underbrace{\max(K - S_{n,j},\, 0)}_{\text{exercise now}}\right)$$

In plain English: at every node, ask "Am I better off holding or exercising?" and take the better option.

> **Key Concept:** This is why the binomial model is so valuable for American options. Unlike BSM, which gives a formula only for European options, the binomial tree naturally handles the early exercise decision at every node. There is no closed-form formula for American put prices --- the binomial tree is one of the standard numerical methods.

In [ ]:
def binomial_american(S0, K, r, T, sigma, N, option_type='put', return_exercise_boundary=False):
    """Price an American option using the CRR binomial model.
    
    At each node, we check: is it better to exercise now or keep holding?
    Optionally returns the early exercise boundary.
    """
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1.0 / u
    q = (np.exp(r * dt) - d) / (u - d)
    disc = np.exp(-r * dt)
    
    # Terminal stock prices
    j = np.arange(N + 1)
    ST = S0 * u**j * d**(N - j)
    
    # Terminal payoffs (same as European at expiry)
    if option_type == 'call':
        V = np.maximum(ST - K, 0.0)
    else:
        V = np.maximum(K - ST, 0.0)
    
    exercise_boundary = np.full(N + 1, np.nan)
    
    # Backward induction WITH early exercise check
    for n in range(N - 1, -1, -1):
        j_arr = np.arange(n + 1)
        S_n = S0 * u**j_arr * d**(n - j_arr)  # stock prices at step n
        
        # Continuation value: what the option is worth if we keep holding
        cont = disc * (q * V[1:n+2] + (1 - q) * V[0:n+1])
        
        # Intrinsic value: what we get if we exercise right now
        if option_type == 'call':
            intrinsic = np.maximum(S_n - K, 0.0)
        else:
            intrinsic = np.maximum(K - S_n, 0.0)
        
        # American option: take the BETTER of exercising now vs. continuing
        V = np.maximum(cont, intrinsic)
        
        # Record the exercise boundary (highest stock price where we exercise a put)
        if return_exercise_boundary:
            exercised = intrinsic > cont + ATOL
            if np.any(exercised):
                if option_type == 'put':
                    exercise_boundary[n] = np.max(S_n[exercised])
                else:
                    exercise_boundary[n] = np.min(S_n[exercised])
    
    if return_exercise_boundary:
        return V[0], exercise_boundary
    return V[0]

Let us compare European and American options to see the **early exercise premium** --- the extra amount an American option is worth due to the early exercise right.

In [ ]:
# Compare European vs American put and call
S0, K, r, T, sigma = 100, 100, 0.05, 1.0, 0.2
N = 500

eur_put = binomial_european(S0, K, r, T, sigma, N, 'put')
am_put  = binomial_american(S0, K, r, T, sigma, N, 'put')
am_call = binomial_american(S0, K, r, T, sigma, N, 'call')
eur_call = binomial_european(S0, K, r, T, sigma, N, 'call')

print(f"European Put: {eur_put:.6f}")
print(f"American Put: {am_put:.6f}  (early exercise premium: {am_put - eur_put:.6f})")
print(f"\nEuropean Call: {eur_call:.6f}")
print(f"American Call: {am_call:.6f}  (premium: {am_call - eur_call:.6f} \u2248 0, as expected)")

As expected:
- The **American put** is worth more than the European put --- the early exercise right has real value (typically \$0.50--\$2.00 for these parameters).
- The **American call** equals the European call --- confirming that early exercise of calls on non-dividend stocks is never optimal.

### The Optimal Exercise Boundary

For the American put, there is a **critical stock price** at each point in time below which you should exercise immediately. This is called the **optimal exercise boundary**.

Think of it like selling a depreciating asset: there comes a point where keeping it costs you more (in lost interest on the strike price you could receive) than any possible future gain from the put becoming even more valuable.

> **Key Concept:** The exercise boundary is not a single number --- it varies with time. Early in the option's life, the stock must be very low before you exercise. Near expiry, even a moderate drop makes early exercise worthwhile because there is almost no time value left.

In [ ]:
# Optimal exercise boundary for American put
N_boundary = 200
am_price, boundary = binomial_american(S0, K, r, T, sigma, N_boundary, 'put',
                                        return_exercise_boundary=True)

time_steps = np.linspace(0, T, N_boundary + 1)
valid = ~np.isnan(boundary)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(time_steps[valid], boundary[valid], color=SECONDARY, linewidth=2, label='Exercise Boundary')
ax.axhline(K, color='gray', linestyle='--', alpha=0.5, label=f'Strike K = {K}')
ax.fill_between(time_steps[valid], 0, boundary[valid], alpha=0.15, color=SECONDARY)
ax.set_xlabel('Time')
ax.set_ylabel('Stock Price')
ax.set_title('Optimal Exercise Boundary for American Put')
ax.legend()
ax.set_xlim(0, T)
plt.tight_layout()
plt.show()

The shaded region is the **exercise region**: if the stock price falls into this zone, you should exercise the put immediately. The boundary rises as expiry approaches --- near expiry, even a modest drop makes early exercise worthwhile because there is little time value left.

> **Important:** The exercise boundary is not a simple number --- it is a function of time. This time-varying boundary is one reason why American put pricing has no closed-form solution and numerical methods like the binomial tree are essential.

## 9. Greeks from the Binomial Tree

The **Greeks** measure how sensitive the option price is to changes in its inputs. They are essential for hedging and risk management.

### What Are Greeks, Intuitively?

| Greek | What It Measures | Plain English |
|-------|-----------------|---------------|
| **Delta** ($\Delta$) | $\partial V / \partial S$ | If the stock moves \$1, how much does my option move? |
| **Gamma** ($\Gamma$) | $\partial^2 V / \partial S^2$ | How fast does delta change? (curvature of the option price) |
| **Theta** ($\Theta$) | $\partial V / \partial t$ | How much value does my option lose each day? (time decay) |

### Real-World Analogies

- **Delta** is like the speedometer of a car: it tells you how fast the option price is moving relative to the stock. A delta of 0.5 means the option is moving at half the speed of the stock.
- **Gamma** is like the acceleration: it tells you how quickly the speedometer reading (delta) is changing. High gamma means delta is changing rapidly --- you need to adjust your hedge frequently.
- **Theta** is like a melting ice cube: every day that passes, the option loses a little value. Time is the enemy of option holders and the friend of option sellers.

### Computing Greeks from the Tree

The binomial tree gives us Greeks naturally through finite differences:

- **Delta**: use the two nodes at step 1 (one up, one down from today).

$$\Delta = \frac{f_{1,1} - f_{1,0}}{S_0 u - S_0 d}$$

- **Gamma**: use the three nodes at step 2.

$$\Gamma = \frac{\frac{f_{2,2} - f_{2,1}}{S_0 u^2 - S_0} - \frac{f_{2,1} - f_{2,0}}{S_0 - S_0 d^2}}{\frac{1}{2}(S_0 u^2 - S_0 d^2)}$$

- **Theta**: compare the option value today with the value at step 2 (same stock price, two time steps later).

$$\Theta = \frac{f_{2,1} - f_{0,0}}{2\Delta t}$$

> **Key Concept:** These are numerical approximations to the true derivatives. They become more accurate as $N$ increases (finer tree). The beauty of the tree approach is that we get Greeks "for free" from the same computation used for pricing.

> **Common Mistake:** Greeks are **instantaneous** sensitivities, valid for small moves. If the stock jumps \$20, you cannot just multiply by delta. You need gamma to account for the curvature. This is like using a tangent line versus the actual curve --- the tangent is only accurate locally.

In [ ]:
def binomial_greeks(S0, K, r, T, sigma, N, option_type='call'):
    """Compute option price, delta, gamma, theta from binomial tree.
    
    We store the first few levels of the tree to extract Greeks.
    """
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1.0 / u
    q = (np.exp(r * dt) - d) / (u - d)
    disc = np.exp(-r * dt)
    
    # Terminal payoffs
    j = np.arange(N + 1)
    ST = S0 * u**j * d**(N - j)
    if option_type == 'call':
        V = np.maximum(ST - K, 0.0)
    else:
        V = np.maximum(K - ST, 0.0)
    
    # Store values at steps 0, 1, 2 for Greek computation
    f = {N: V.copy()}
    
    for n in range(N - 1, -1, -1):
        V = disc * (q * V[1:n+2] + (1 - q) * V[0:n+1])
        if n <= 2:
            f[n] = V.copy()
    
    price = f[0][0]
    
    # Delta: slope from step 1
    delta = (f[1][1] - f[1][0]) / (S0 * u - S0 * d)
    
    # Gamma: curvature from step 2
    h1 = S0 * u**2 - S0
    h2 = S0 - S0 * d**2
    h3 = 0.5 * (S0 * u**2 - S0 * d**2)
    gamma = ((f[2][2] - f[2][1]) / h1 - (f[2][1] - f[2][0]) / h2) / h3
    
    # Theta: time decay from step 2
    theta = (f[2][1] - f[0][0]) / (2 * dt)
    
    return {'price': price, 'delta': delta, 'gamma': gamma, 'theta': theta}


# Compute Greeks for call and put
N = 500
greeks_call = binomial_greeks(S0, K, r, T, sigma, N, 'call')
greeks_put  = binomial_greeks(S0, K, r, T, sigma, N, 'put')

print("Binomial Greeks (N=500):")
print(f"{'':15s} {'Call':>12s} {'Put':>12s}")
for g in ['price', 'delta', 'gamma', 'theta']:
    print(f"  {g:13s} {greeks_call[g]:12.6f} {greeks_put[g]:12.6f}")

### Interpreting the Greeks

For our at-the-money call ($S = K = 100$):

- **Delta $\approx 0.64$**: If the stock rises by \$1, the call increases by about \$0.64. The call is not a 1-for-1 bet on the stock.
- **Gamma $\approx 0.019$**: Delta changes by about 0.019 per \$1 move in the stock. Gamma is highest for ATM options.
- **Theta $\approx -6.4$**: The call loses about \$6.40 per year (or roughly \$0.025 per day) just from the passage of time. Time is the enemy of option holders.

For the put:
- **Delta $\approx -0.36$**: The put moves opposite to the stock (negative delta). A \$1 rise in the stock decreases the put by about \$0.36.
- **Gamma is the same** as the call's gamma. This follows from put-call parity: $\Delta_C - \Delta_P = 1$, so $\Gamma_C = \Gamma_P$.

> **Key Concept:** A trader who is "delta neutral" (delta = 0) still has gamma exposure. Gamma is what makes options non-linear instruments. It is both the opportunity and the risk of options trading.

## 10. Visualizing the Binomial Tree

A picture is worth a thousand formulas. Let us draw a small binomial tree showing stock prices (top of each node) and option values (bottom, in color). This makes the backward induction process visible.

Look at how each interior node's option value is the discounted weighted average of its two children. Trace from the terminal payoffs backwards to see the price emerge at the root.

In [ ]:
def plot_binomial_tree(S0, K, r, T, sigma, N, option_type='call'):
    """Plot a binomial tree with stock prices and option values."""
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1.0 / u
    q = (np.exp(r * dt) - d) / (u - d)
    disc = np.exp(-r * dt)
    
    # Build stock price tree
    S = []
    for n in range(N + 1):
        j = np.arange(n + 1)
        S.append(S0 * u**j * d**(n - j))
    
    # Terminal payoffs
    if option_type == 'call':
        V_tree = [np.maximum(S[N] - K, 0.0)]
    else:
        V_tree = [np.maximum(K - S[N], 0.0)]
    
    # Backward induction, storing all values
    V = V_tree[0].copy()
    for n in range(N - 1, -1, -1):
        V = disc * (q * V[1:n+2] + (1 - q) * V[0:n+1])
        V_tree.insert(0, V.copy())
    
    # Plot
    fig, ax = plt.subplots(figsize=(max(12, 3*N), max(8, 2*N)))
    
    for n in range(N + 1):
        for j in range(n + 1):
            y = j - n / 2  # center the tree
            ax.plot(n, y, 'o', color=PRIMARY, markersize=30, zorder=3)
            ax.text(n, y + 0.15, f'S={S[n][j]:.1f}', ha='center', va='bottom', fontsize=7, fontweight='bold')
            ax.text(n, y - 0.15, f'V={V_tree[n][j]:.2f}', ha='center', va='top', fontsize=7, color=SECONDARY)
            
            if n < N:
                ax.plot([n, n+1], [y, j+1 - (n+1)/2], '-', color='gray', alpha=0.5, zorder=1)
                ax.plot([n, n+1], [y, j - (n+1)/2], '-', color='gray', alpha=0.5, zorder=1)
    
    ax.set_xlabel('Time Step')
    ax.set_title(f'Binomial Tree: European {option_type.title()} (N={N})')
    ax.set_xticks(range(N + 1))
    ax.grid(False)
    plt.tight_layout()
    plt.show()


plot_binomial_tree(S0=100, K=100, r=0.05, T=1.0, sigma=0.2, N=4, option_type='call')

In this tree visualization, you can trace the backward induction process. Each interior node's option value (V) is the discounted weighted average of the two nodes it connects to at the next step. The terminal nodes (rightmost) show the raw payoff $\max(S - K, 0)$.

> **Key Concept:** The tree makes the option pricing algorithm tangible. You can literally see the price "flowing" backwards from the known terminal payoffs to the unknown present value. No other pricing method offers this level of visual transparency.

## 11. Convergence Analysis: Even-Odd Oscillation and Richardson Extrapolation

### Why Does the Binomial Price Zigzag?

If you look closely at the convergence plot, even $N$ and odd $N$ approach the BSM value from opposite sides. This is not a bug --- it is a feature of the lattice structure.

The oscillation occurs because for even $N$, the strike price $K$ may fall exactly on a node of the terminal distribution, while for odd $N$ it falls between nodes (or vice versa). This creates a systematic alternating bias.

### Richardson Extrapolation: A Clever Fix

Since the error alternates sign, we can cancel most of it by averaging consecutive estimates:

$$V_{\text{extrapolated}} = \frac{V(N) + V(N+1)}{2}$$

This simple trick improves convergence from $O(1/N)$ to $O(1/N^2)$ --- a dramatic improvement.

> **Key Concept:** Richardson extrapolation exploits the structure of the error to accelerate convergence. It is a general technique that works whenever the error has a known form (here, alternating sign). In practice, it gives you the accuracy of a tree with many more steps, essentially for free.

In [ ]:
# Even-odd oscillation and Richardson extrapolation
N_values = np.arange(10, 201)
prices = np.array([binomial_european(S0, K, r, T, sigma, int(N), 'call') for N in N_values])
bsm_ref = bsm_price(S0, K, r, T, sigma, 'call')

# Richardson extrapolation: average of consecutive values
richardson = 0.5 * (prices[:-1] + prices[1:])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw prices with even/odd coloring
even_mask = N_values % 2 == 0
axes[0].scatter(N_values[even_mask], prices[even_mask], s=5, color=PRIMARY, alpha=0.6, label='Even N')
axes[0].scatter(N_values[~even_mask], prices[~even_mask], s=5, color=SECONDARY, alpha=0.6, label='Odd N')
axes[0].axhline(bsm_ref, color='black', linestyle='--', linewidth=1, label=f'BSM = {bsm_ref:.4f}')
axes[0].set_xlabel('N')
axes[0].set_ylabel('Price')
axes[0].set_title('Even-Odd Oscillation')
axes[0].legend()

# Right: error comparison
axes[1].semilogy(N_values, np.abs(prices - bsm_ref), color=PRIMARY, alpha=0.5, linewidth=0.8, label='Raw')
axes[1].semilogy(N_values[:-1] + 0.5, np.abs(richardson - bsm_ref), color=SECONDARY, alpha=0.7, linewidth=1.2,
                 label='Richardson')
axes[1].set_xlabel('N')
axes[1].set_ylabel('Absolute Error')
axes[1].set_title('Richardson Extrapolation')
axes[1].legend()

plt.tight_layout()
plt.show()

The Richardson-extrapolated values (orange line in the right panel) have dramatically smaller errors than the raw binomial prices. The error drops roughly two orders of magnitude, giving us the accuracy of a tree with many more steps.

### Summary of Convergence Rates

| Method | Error Rate | Practical Implication |
|--------|------------|---------------------|
| Raw CRR binomial | $O(1/N)$ | Need ~1000 steps for 4-digit accuracy |
| Richardson extrapolation | $O(1/N^2)$ | Need ~30 steps for 4-digit accuracy |
| BSM formula (exact) | 0 | Instant, but only works for European options |

> **Important:** Richardson extrapolation is practically free --- it requires only two tree evaluations (for $N$ and $N+1$) and one average. Always use it when applying the binomial model for European options.

## 12. When to Use the Binomial Model

Given that the BSM formula exists and is exact, why bother with the binomial model?

| Situation | Use Binomial? | Why |
|-----------|:---:|-----|
| European option, constant vol | No | BSM formula is exact and instant |
| American option | **Yes** | No closed-form solution exists |
| Dividends at known dates | **Yes** | Easy to adjust the tree at dividend dates |
| Pedagogical understanding | **Yes** | The tree makes no-arbitrage pricing visible |
| Exotic path-dependent options | Maybe | Monte Carlo is often more flexible |
| Very high accuracy needed | Depends | Tree with Richardson can match BSM well |

The binomial model occupies a sweet spot between simplicity and generality. It is the go-to method for American options and remains a cornerstone of practical derivatives pricing.

> **Key Concept:** The binomial model is not just a stepping stone to BSM --- it is a practical tool in its own right. Its ability to handle early exercise, discrete dividends, and time-varying parameters makes it indispensable. Every quant should be fluent in binomial trees.

## 13. References

1. Cox, J., Ross, S., & Rubinstein, M. (1979). *Option pricing: A simplified approach*. Journal of Financial Economics, 7(3), 229-263.
2. Hull, J. C. (2018). *Options, Futures, and Other Derivatives* (10th ed.). Pearson.
3. Shreve, S. E. (2004). *Stochastic Calculus for Finance I: The Binomial Asset Pricing Model*. Springer.
4. Broadie, M., & Detemple, J. (1996). *American option valuation: New bounds, approximations, and a comparison of existing methods*. Review of Financial Studies, 9(4), 1211-1250.
5. Leisen, D. P. J., & Reimer, M. (1996). *Binomial models for option valuation --- examining and improving convergence*. Applied Mathematical Finance, 3(4), 319-346.